# NB04 - Learning Rate Schedules
## Background
A fixed learning rate is almost never optimal. Learning rate schedules systematically adjust lr throughout training. Modern practice (particularly for transformers) uses warmup + cosine decay, popularized by the original BERT paper. The one-cycle policy (Smith & Topin, 2018) enables super-convergence in CNNs. The LR range test (Smith, 2017) finds the optimal lr range empirically before committing to a training run.

## What You'll Learn
- Warmup+cosine, cosine with warm restarts (SGDR), linear decay
- One-cycle policy with superconvergence properties
- Step decay and the original transformer schedule
- LR range test: finding optimal lr range empirically

## Why This Matters
The LR schedule is often the difference between a model that trains well and one that diverges. Warmup is critical for transformers: without it, the large initial gradient variance destabilizes early training. The transformer original schedule peaked at step 4000 for d_model=512.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def warmup_cosine(step, total_steps, warmup_steps, lr_max, lr_min=0.0):
    if step < warmup_steps:
        return lr_max * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

def cosine_with_restarts(step, T_0=200, T_mult=2, lr_max=1e-3, lr_min=1e-6):
    t, T_cur = step, T_0
    while t >= T_cur:
        t -= T_cur
        T_cur = T_cur * T_mult
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T_cur))

def linear_decay(step, total_steps, warmup_steps, lr_max, lr_min=0.0):
    if step < warmup_steps:
        return lr_max * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(lr_max * (1 - progress), lr_min)

def one_cycle(step, total_steps, lr_max, lr_min_ratio=0.1, pct_start=0.3):
    warmup_steps = int(total_steps * pct_start)
    lr_init = lr_max * lr_min_ratio
    if step <= warmup_steps:
        return lr_init + (lr_max - lr_init) * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    lr_floor = lr_min_ratio**2 * lr_max
    return lr_floor + 0.5*(lr_max - lr_floor)*(1 + np.cos(np.pi * progress))

def step_decay(step, lr_max, drop_every=100, drop_factor=0.5):
    return lr_max * (drop_factor ** (step // drop_every))

total_steps = 1000
steps = np.arange(total_steps)
lr_max = 1e-3

schedules = {
    'Warmup+Cosine (100 warmup)': [warmup_cosine(s, total_steps, 100, lr_max) for s in steps],
    'Cosine+Restarts (T0=200, Tmult=2)': [cosine_with_restarts(s, 200, 2, lr_max) for s in steps],
    'Warmup+Linear (100 warmup)': [linear_decay(s, total_steps, 100, lr_max) for s in steps],
    'One-Cycle (30% warmup)': [one_cycle(s, total_steps, lr_max) for s in steps],
    'Step Decay (every 200 steps)': [step_decay(s, lr_max, 200, 0.5) for s in steps],
}

fig, ax = plt.subplots(figsize=(12, 6))
for name, lrs in schedules.items():
    ax.plot(steps, lrs, label=name, linewidth=1.5)
ax.set_title('Learning Rate Schedules')
ax.set_xlabel('Step'); ax.set_ylabel('Learning Rate')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_04_lr_schedules.png', dpi=100, bbox_inches='tight')
plt.show()
print('All schedules visualized.')

In [ ]:
np.random.seed(42)

lr_range = np.logspace(-7, 0, 100)
losses = []
for lr in lr_range:
    if lr < 1e-5:
        loss = 2.0 - lr * 1e4 * 0.1 + 0.05 * np.random.randn()
    elif lr < 1e-2:
        loss = 2.0 - 0.1 * np.log10(lr / 1e-5) + 0.05 * np.random.randn()
    else:
        loss = 2.0 + (lr / 1e-2)**1.5 + 0.1 * np.random.randn()
    losses.append(max(0.1, loss))
losses = np.array(losses)

dloss = np.gradient(losses)
best_lr_idx = np.argmin(dloss)
best_lr = lr_range[best_lr_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogx(lr_range, losses, 'b-', linewidth=2)
axes[0].axvline(best_lr, color='red', linestyle='--', label=f'Optimal LR ~ {best_lr:.2e}')
axes[0].set_title('LR Range Test'); axes[0].set_xlabel('Learning Rate (log scale)')
axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogx(lr_range, dloss, 'g-', linewidth=2)
axes[1].axvline(best_lr, color='red', linestyle='--', label=f'Min slope at {best_lr:.2e}')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Loss Gradient'); axes[1].set_xlabel('Learning Rate (log scale)')
axes[1].set_ylabel('dLoss/dLR'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_04_lr_range_test.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Suggested LR from range test: {best_lr:.2e}')
print(f'Conservative choice (1/10x): {best_lr/10:.2e}')
print()
print('Why transformers need warmup:')
print('  At step 0, v ~ 0, so effective_lr = lr / (sqrt(0) + eps) -> huge')
print(f'  For beta2=0.999: v stabilizes after ~{1/(1-0.999):.0f} steps')
print('  Warmup keeps lr small until v has accumulated real gradient info')